# Sell in May and Go Away: a Python backtest across 7 markets

Education only. This is not investment advice, a recommendation or a forecast, and it is not a claim about returns or performance. Backtests are hypothetical, use past data, ignore many real-world factors and do not predict future results. I am not a SEBI-registered investment adviser or research analyst.

This notebook tests the Nov-Apr vs May-Oct rule on India, the US, UK, France, Japan, Hong Kong and China, using the same rule and the same code for every market. One market (USA) is shown in full detail; the rest run through the same pipeline and land in one scoreboard.

**Rules are locked before looking at results (Cell 1). They are not changed afterward.**

In [ ]:
# ============================================================
# CELL 1: Setup, imports, and settings — LOCKED before results
# ============================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


ROUND_TRIP_COST = 0.003          # 0.3% per buy+sell

# --- Markets to test: name -> actual ticker symbol Yahoo Finance needs ---
MARKETS = {
    "India (Nifty 50)":        "^NSEI",
    "USA (S&P 500)":           "^GSPC",
    "UK (FTSE 100)":           "^FTSE",
    "France (CAC 40)":         "^FCHI",
    "Japan (Nikkei 225)":      "^N225",
    "Hong Kong (Hang Seng)":   "^HSI",
    "China (SSE Composite)":   "000001.SS",
}

CASH_RATE_BY_MARKET = {
    "India (Nifty 50)":        0.06,   # India short-term rates, roughly
    "USA (S&P 500)":           0.02,
    "UK (FTSE 100)":           0.02,
    "France (CAC 40)":         0.01,   # Eurozone rates
    "Japan (Nikkei 225)":      0.002,  # near-zero for most of the period
    "Hong Kong (Hang Seng)":   0.02,
    "China (SSE Composite)":   0.025,
}

DETAIL_MARKET = "USA (S&P 500)"   
START = "2000-01-01"             
                                
START_CAPITAL = 100_000

print(f"Settings locked. Testing {len(MARKETS)} markets. No tuning after results.")

In [ ]:
# ============================================================
# CELL 2: Download every market (up to 31 July 2026) + visualize
# ============================================================

import yfinance as yf

END = "2026-08-01"   # exclusive — this includes 31 July 2026

raw_close = {}
for name, ticker in MARKETS.items():
    try:
        df = yf.download(
            ticker,
            start=START,
            end=END,
            auto_adjust=True,
            progress=False,
        )
        if df is None or df.empty:
            print(f"{name:26s} No data returned.")
            continue

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        s = df["Close"].astype(float).dropna().sort_index()
        s = s[~s.index.duplicated()]
        raw_close[name] = s
        print(f"{name:26s} {s.index.min().date()} to {s.index.max().date()}  ({len(s)} rows)")

    except Exception as e:
        print(f"{name:26s} Failed to download data: {e}")

# --- Visualize: number of trading days per market (bar chart) ---
row_counts = {name: len(s) for name, s in raw_close.items()}

fig = go.Figure(go.Bar(
    x=list(row_counts.keys()),
    y=list(row_counts.values()),
    text=[f"{v:,}" for v in row_counts.values()],
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(color="white"),
    marker_color="#1E88E5",
))

for name, v in row_counts.items():
    fig.add_annotation(
        x=name, y=v, text=f"~{v/252:.0f} yrs",
        showarrow=False, yshift=14,
        font=dict(size=13, color="black"),
    )

fig.update_layout(
    title="Number of trading days available, by market",
    yaxis_title="Rows (trading days)",
    template="plotly_white",
    height=480,
    xaxis_tickangle=-20,
)
fig.show()

In [ ]:
# ============================================================
# CELL 3: Data check for every market
# ============================================================

def data_check(name, close):
    rets = close.pct_change().dropna()
    big = (rets.abs() > 0.08).sum()
    print(f"{name:26s} rows={len(close):5d}  big-move days(>8%)={big:2d}")
    return big

print("=== DATA CHECK (all markets) ===")
big_moves = {}
for name, s in raw_close.items():
    big_moves[name] = data_check(name, s)

# --- Visualize: big-move days per market (bar chart) ---
fig = go.Figure(go.Bar(
    x=list(big_moves.keys()),
    y=list(big_moves.values()),
    text=list(big_moves.values()),
    textposition="outside",
    marker_color="#EF4444",
))

fig.update_layout(
    title="Days with a move bigger than 8%, by market",
    yaxis_title="Number of days",
    template="plotly_white",
    height=480,
    xaxis_tickangle=-20,
)
fig.show()

In [ ]:
# ============================================================
# CELL 4: Detail market — May analysis + chart (combined)
# ============================================================

close = raw_close[DETAIL_MARKET]
monthly_ret = close.resample("ME").last().pct_change() * 100
may = monthly_ret[monthly_ret.index.month == 5].dropna()
other = monthly_ret[monthly_ret.index.month != 5].dropna()

# --- Print the summary ---
print(f"=== {DETAIL_MARKET}: MAY-ONLY ANALYSIS ===")
print(f"Number of Mays: {len(may)}")
print(f"Mays positive: {(may > 0).sum()}  ({(may > 0).mean()*100:.1f}%)")
print(f"Average May return: {may.mean():.2f}%   Other months average: {other.mean():.2f}%")
print(f"Best May: {may.max():.2f}% ({may.idxmax().year})   Worst May: {may.min():.2f}% ({may.idxmin().year})")

# --- Same data as a chart ---
colors = ["#2e7d32" if x > 0 else "#c62828" for x in may.values]
fig = go.Figure(go.Bar(
    x=may.index.year, y=may.values, marker_color=colors,
    text=[f"{v:.1f}%" for v in may.values], textposition="outside",
    hovertemplate="<b>Year %{x}</b><br>May return: %{y:.2f}%<extra></extra>"
))
fig.add_hline(y=0, line_color="black", line_width=1)
fig.update_layout(
    title=f"{DETAIL_MARKET}: May returns by year",
    xaxis_title="Year", yaxis_title="Return (%)",
    template="plotly_white", height=460, showlegend=False,
)
fig.show()

In [ ]:
# ============================================================
# CELL 5: Backtest functions (shared by every market)
# ============================================================

def build_positions(close):
    months = close.index.month
    in_season = (months >= 11) | (months <= 4)
    return {
        "Sell in May (Nov-Apr only)": pd.Series(in_season.astype(float), index=close.index),
        "Buy and hold": pd.Series(1.0, index=close.index),
    }

def net_returns(ret, held, cash_rate):
    turnover = held.diff().abs()
    turnover.iloc[0] = abs(held.iloc[0])
    cash_daily = (1 + cash_rate) ** (1 / 252) - 1
    return held * ret + (1 - held) * cash_daily - turnover * (ROUND_TRIP_COST / 2)

def metrics(net, held):
    equity = (1 + net).cumprod()
    years = (net.index[-1] - net.index[0]).days / 365.25
    cagr = equity.iloc[-1] ** (1 / years) - 1
    max_dd = (equity / equity.cummax() - 1).min()
    sharpe = net.mean() / net.std() * np.sqrt(252) if net.std() > 0 else np.nan
    entries = held.diff().fillna(held) == 1
    return {
        "Final value": round(START_CAPITAL * equity.iloc[-1]),
        "CAGR %": round(cagr * 100, 2),
        "Max drawdown %": round(max_dd * 100, 1),
        "Sharpe": round(sharpe, 2),
        "Trades": int(entries.sum()),
        "Time invested %": round(held.mean() * 100, 1),
    }, equity

print("Functions defined. Identical for every market.")

In [ ]:
# ============================================================
# CELL 6: Run the backtest for EVERY market
# ============================================================

results = {}     # market -> {strategy: metrics dict}
curves = {}      # market -> {strategy: equity series}

for name, close_m in raw_close.items():
    cash_rate = CASH_RATE_BY_MARKET[name]
    ret = close_m.pct_change()
    positions = build_positions(close_m)
    held_all = {k: v.shift(2).fillna(0.0) for k, v in positions.items()}

    # no warmup — a calendar rule needs no indicator history
    idx = ret.index

    rows, eqs = {}, {}
    for strat_name, held in held_all.items():
        m, eq = metrics(net_returns(ret.loc[idx], held.loc[idx], cash_rate), held.loc[idx])
        rows[strat_name] = m
        eqs[strat_name] = eq
    results[name] = rows
    curves[name] = eqs

print(f"Backtest complete for {len(results)} markets.")

In [ ]:
# ============================================================
# CELL 7: The scoreboard
# ============================================================

scoreboard_rows = []
for name, rows in results.items():
    sim = rows["Sell in May (Nov-Apr only)"]
    bah = rows["Buy and hold"]
    scoreboard_rows.append({
        "Market": name,
        "Sell in May CAGR %": sim["CAGR %"],
        "Buy & Hold CAGR %": bah["CAGR %"],
        "Gap (SiM - B&H)": round(sim["CAGR %"] - bah["CAGR %"], 2),
        "SiM Max DD %": sim["Max drawdown %"],
        "B&H Max DD %": bah["Max drawdown %"],
        "Beat buy-and-hold?": "Yes" if sim["CAGR %"] > bah["CAGR %"] else "No",
    })

scoreboard = pd.DataFrame(scoreboard_rows).sort_values("Gap (SiM - B&H)", ascending=False)
print("=== SCOREBOARD: Sell in May vs Buy-and-Hold, by market ===")
print(scoreboard.to_string(index=False))

win_count = (scoreboard["Beat buy-and-hold?"] == "Yes").sum()
print(f"\nSell in May beat buy-and-hold in {win_count} out of {len(scoreboard)} markets.")

In [ ]:
# ============================================================
# CELL 8: Scoreboard chart
# ============================================================

fig = go.Figure()
fig.add_trace(go.Bar(name="Sell in May", x=scoreboard["Market"], y=scoreboard["Sell in May CAGR %"],
                      marker_color="#F5B301"))
fig.add_trace(go.Bar(name="Buy and hold", x=scoreboard["Market"], y=scoreboard["Buy & Hold CAGR %"],
                      marker_color="#1E88E5"))
fig.update_layout(barmode="group", title="CAGR: Sell in May vs Buy-and-Hold, by market",
                   yaxis_title="CAGR (%)", template="plotly_white", height=520,
                   xaxis_tickangle=-20)
fig.show()

In [ ]:
# ============================================================
# CELL 8b: Drawdown chart — Max drawdown by market
# ============================================================
fig = go.Figure()
fig.add_trace(go.Bar(name="Sell in May", x=scoreboard["Market"], y=scoreboard["SiM Max DD %"],
                      marker_color="#F5B301"))
fig.add_trace(go.Bar(name="Buy and hold", x=scoreboard["Market"], y=scoreboard["B&H Max DD %"],
                      marker_color="#1E88E5"))
fig.update_layout(barmode="group", title="Max Drawdown: Sell in May vs Buy-and-Hold, by market",
                   yaxis_title="Max Drawdown (%)", template="plotly_white", height=520,
                   xaxis_tickangle=-20)
fig.show()

In [ ]:
# Cell 9: Detail market, relative growth of each rule
fig = go.Figure()
for strat_name, eq in curves[DETAIL_MARKET].items():
    fig.add_trace(go.Scatter(x=eq.index, y=eq, mode="lines", name=strat_name,
                             line=dict(width=3 if strat_name == "Buy and hold" else 2)))
fig.update_layout(title=f"{DETAIL_MARKET}: relative growth of each rule (start = 1, log scale)",
                  yaxis_type="log", template="plotly_white", height=520, hovermode="x unified")
fig.show()

In [ ]:
import pickle
cash_rates = {m: float(CASH_RATE_BY_MARKET[m]) for m in results.keys()}
print(cash_rates)
with open("cash_rates.pkl", "wb") as f:
    pickle.dump(cash_rates, f)